# Phase 4 CJPE — 02 B1 Partial

One independent T4 notebook. All source is embedded. This notebook can start without any other notebook finishing.

Required own checkpoint bundles: **Phase3_B1_Input.zip**. The Phase3 filename is historical: these contain original Phase1 LM weights, matching tokenizers and manifests. No Phase2 NER checkpoint is needed.

## Launch this notebook in its own account/runtime
1. In that account's Drive create **My Drive / CS781_Phase4 / inputs**.
2. Upload these existing Windows files there, without extracting:
- `D:\CS781\Phase4_20261020\inputs\Phase3_B1_Input.zip`
3. Select **T4 GPU**. Run the setup cells in order, approve Drive using the same Google account, and sign into your own Hugging Face account with course-dataset access when requested. Finish authentication before leaving it running.
4. Run CONFIG and **Start/resume**. That cell checks GPU gradients, prepares LOCAL SQLite caches, trains, validates, copies the submission ZIP into **My Drive / CS781_Phase4 / SUBMISSIONS**, then packages model/source records. No separate status/packaging cell is required.
5. Confirm status reaches **training** with increasing optimizer steps. Indexing/tokenizing and a preflight with optimizer_steps0 are preparation stages.

All three observed fixes are embedded: identical train/dev copies counted once, original Unicode13 tokenizer compatibility, and SQLite caches on local runtime disk. Do not add any old repair cells to this fresh notebook.

Select **Runtime → Change runtime type → T4 GPU**. Use only corrected official CJPE data. Login/mount approvals need your action before leaving it running. Actual GPU time, scores and schema compatibility have not been established in the authoring environment.

Persistent checkpoints/results: `MyDrive/CS781_Phase4/runs/02_B1_Partial/`. Every result ZIP contains only root prediction.json. MODEL_RECORDS ZIPs are for your records and must not be submitted to Codabench.

Deadline: **21 October 2026, 23:59 IST**. Default training stops early enough to reserve 45 minutes for export. An early result is exported after the first holdout evaluation; use the final result when it is ready. No automated Codabench uploads.


In [1]:
# Keep Colab's CUDA-enabled torch and existing tokenizers; do not install FlashAttention or model weights.
%pip -q install "ijson>=3.2,<4" "huggingface_hub>=0.26,<2" "scikit-learn>=1.4,<2"
import importlib.util,subprocess,sys
if importlib.util.find_spec('tokenizers') is None:
    subprocess.check_call([sys.executable,'-m','pip','install','-q','tokenizers'])
import torch,platform,pathlib,shutil
print('Python:',platform.python_version(),'Torch:',torch.__version__)
assert torch.cuda.is_available(), 'Select a GPU runtime first.'
print('ACTUAL GPU:',torch.cuda.get_device_name(0))
print('CUDA:',torch.version.cuda,'free local GB:',round(shutil.disk_usage('/').free/1e9,2))


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.2/150.2 kB 13.6 MB/s eta 0:00:00
Python: 3.13.15 Torch: 2.11.0+cu130
ACTUAL GPU: Tesla T4
CUDA: 13.0 free local GB: 75.46


In [2]:
# Embedded source with exact SHA verification; no separate source upload.
import base64,hashlib,io,zipfile,pathlib,sys
source_bytes=base64.b64decode('...')
assert hashlib.sha256(source_bytes).hexdigest()=='0d776ceb1277195ad3c1f59b2c0a83d3b0d180b6ac3641d3196547e0ab309dbb'
CODE_ROOT=pathlib.Path('/content/phase4_code' if pathlib.Path('/content').exists() else '/kaggle/working/phase4_code')
CODE_ROOT.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(io.BytesIO(source_bytes)) as z:
    for item in z.infolist():
        p=pathlib.PurePosixPath(item.filename)
        assert not p.is_absolute() and '..' not in p.parts
        dest=CODE_ROOT/item.filename;dest.parent.mkdir(parents=True,exist_ok=True);dest.write_bytes(z.read(item))
if str(CODE_ROOT) not in sys.path:sys.path.insert(0,str(CODE_ROOT))
print('Source ready:',CODE_ROOT)


Source ready: /content/phase4_code


In [3]:
# Choose how your own checkpoint bundles are supplied.
from pathlib import Path
INPUT_MODE='drive'  # 'drive' or 'temporary_upload'
if Path('/content').exists():
    from google.colab import drive
    drive.mount('/content/drive')
    PERSIST=Path('/content/drive/MyDrive/CS781_Phase4')
    WORK=Path('/content/phase4_work')
else:
    PERSIST=Path('/kaggle/working/CS781_Phase4');WORK=PERSIST/'work'
PERSIST.mkdir(parents=True,exist_ok=True);WORK.mkdir(parents=True,exist_ok=True)
INPUT_DIR=PERSIST/'inputs' if INPUT_MODE=='drive' else Path('/content/inputs')
# For an already-uploaded Phase3 bundle you may set INPUT_DIR to that Drive folder.
REQUIRED=['Phase3_B1_Input.zip']
EXPECTED={'Phase3_B1_Input.zip': {'bytes': 1491952478, 'sha256': '22b49e8f370e8f9bfc1a3f3f88cab813c0e7443edb9091e30f8d4877508690c9'}}
from phase4_cjpe.common import sha
for filename in REQUIRED:
    path=INPUT_DIR/filename
    # Optional support for the exact numbered parts already uploaded in Phase3.
    if not path.exists():
        parts=sorted(INPUT_DIR.glob(filename+'.part*'))
        if parts:
            assert [p.name for p in parts]==[filename+f'.part{i:03d}' for i in range(1,len(parts)+1)], 'Incomplete numbered input parts'
            path=WORK/filename
            with path.with_suffix('.partial').open('wb') as out:
                for part in parts:
                    with part.open('rb') as src:shutil.copyfileobj(src,out,8*1024*1024)
            path.with_suffix('.partial').replace(path)
    assert path.exists(), f'Upload {filename} to {INPUT_DIR} using Drive, or the Colab Files pane for temporary mode.'
    assert path.stat().st_size==EXPECTED[filename]['bytes'], 'Incomplete checkpoint upload'
    assert sha(path)==EXPECTED[filename]['sha256'], 'Checkpoint bundle SHA mismatch'
    print('Verified input:',path)
    EXPECTED[filename]['resolved_path']=str(path)


ValueError: mount failed

In [ ]:
# Official corrected data: uploaded ZIP, existing folder, or explicit approved HF login.
from phase4_cjpe.data import acquire,index_data
DATA_ROOT=WORK/'Phase_4_CJPE'
OFFICIAL_DATA_ZIP=None  # Example: str(INPUT_DIR/'Phase_4_CJPE.zip')
# Alternatively upload Train/train.json, Train/dev.json, Test/test.json to DATA_ROOT.
missing=any(not (DATA_ROOT/relative).exists() for relative in ('Train/train.json','Train/dev.json','Test/test.json'))
if missing and not OFFICIAL_DATA_ZIP:
    from huggingface_hub import notebook_login
    notebook_login()  # Authenticate only to your own account with approved official-dataset access.
DATA_ROOT=acquire(DATA_ROOT,uploaded_zip=OFFICIAL_DATA_ZIP,download=bool(missing and not OFFICIAL_DATA_ZIP))
print('Official data ready:',DATA_ROOT)
print('The worker will stream records and check labels/IDs and cross-split text duplication before training.')


In [ ]:
# Configure BEFORE first run; keep it unchanged when resuming the same RUN_ROOT.
CONFIG={'name': '02_B1_Partial', 'family': 'b1', 'method': 'partial', 'experiment': 'adaptation', 'rank': 8, 'alpha': 16, 'seed': 781, 'max_tokens': 256, 'chunks_per_case': 4, 'region_chars': 8192, 'max_train_cases': 12000, 'holdout_per_class': 256, 'max_epochs': 4, 'accumulation': 8, 'backbone_lr': 2e-05, 'head_lr': 0.0003, 'training_hours': 2.5, 'eval_every': 100, 'save_every': 25, 'patience': 5, 'teacher_hours': 1.5, 'kd_temperature': 2.0, 'kd_weight': 0.5, 'stall_minutes': 45, 'export_reserve_minutes': 45, 'deadline_utc': '2026-10-21T18:29:00Z', 'allow_after_deadline': False}
RUN_ROOT=PERSIST/'runs'/'02_B1_Partial'
CONFIG.update(run_root=str(RUN_ROOT),work_root=str(WORK),data_root=str(DATA_ROOT),
              bundle=EXPECTED['Phase3_B1_Input.zip']['resolved_path'])

print(CONFIG)
print('12,000 is the initial TRAINING-CASE cap, not a token count. Use None for the full official train split in a new run.')
print('Nominal training budget:',CONFIG['training_hours'],'hours; teacher adds',CONFIG['teacher_hours'] if CONFIG['experiment']=='distillation' else 0)


In [ ]:
# START/RESUME: GPU preflight, training, submission export and model/source records.
import copy,gc,json,shutil,time
from pathlib import Path
import torch
from IPython.display import FileLink,display
from phase3_rr.archive_inputs import resolve_bundle
from phase3_rr.data import Tokenizer
from phase4_cjpe.common import atomic_json,sha,status,source_hashes,now
from phase4_cjpe.models import make_model
from phase4_cjpe.runtime import supervise
from phase4_cjpe.export import validate_zip,package_records

assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU.'
RUN_ROOT=Path(CONFIG['run_root']);RUN_ROOT.mkdir(parents=True,exist_ok=True)
for process in Path('/proc').iterdir():
    if not process.name.isdigit():continue
    try:command=process.joinpath('cmdline').read_bytes().split(b'\0')
    except (FileNotFoundError,PermissionError,ProcessLookupError):continue
    if b'phase4_cjpe.runtime' in command and b'--worker' in command:
        raise RuntimeError('A Phase4 worker is already active in this runtime. PID='+process.name)

def _preflight(cfg,stage):
    status(RUN_ROOT,'gpu_preflight',error=None,note=stage+'; optimizer training has not started.')
    bundle,manifest=resolve_bundle(cfg['bundle'],Path(cfg['work_root'])/'weights')
    assert manifest['family']==cfg['family'], 'Wrong input checkpoint family'
    original_sha=sha(bundle/'tokenizer.json')
    tok=Tokenizer(bundle/'tokenizer.json')
    metadata=json.loads((bundle/'tokenizer.json').read_text())
    text='The Supreme Court considers the appeal under section १२३.'
    tokens=tok.encode(text)
    assert tokens, 'Tokenizer produced no tokens'
    if metadata.get('format')=='cs781-own-byte-bpe-v1':
        assert tok.impl.decode(tokens)==text, 'Original own tokenizer round-trip failed'
    assert sha(bundle/'tokenizer.json')==original_sha, 'Original tokenizer changed'
    model=None
    try:
        model,evidence=make_model(bundle,cfg['family'],cfg['method'],cfg['rank'],cfg['alpha'])
        model=model.to('cuda').train()
        ids=[1]+tokens[:30]+[2]
        with torch.autocast(device_type='cuda',dtype=torch.float16):
            logits=model([ids],[0.0],torch.device('cuda')).float()
        loss=torch.nn.functional.cross_entropy(logits,torch.tensor([0],device='cuda'))
        assert bool(torch.isfinite(loss)), 'Nonfinite GPU preflight loss'
        scaler=torch.amp.GradScaler('cuda',init_scale=128)
        scaler.scale(loss).backward()
        grads=[p.grad for p in model.backbone.parameters() if p.requires_grad and p.grad is not None]
        assert grads and all(bool(torch.isfinite(g).all()) for g in grads), 'Nonfinite/missing backbone gradients'
        assert any(bool(g.abs().max()>0) for g in grads), 'No nonzero backbone/adaptation gradients'
        record={'stage':stage,'checked_utc':now(),'gpu':torch.cuda.get_device_name(0),
                'family':cfg['family'],'method':cfg['method'],'finite_loss':float(loss.detach()),
                'parameters':evidence['parameters'],'trainable_parameters':evidence['trainable_parameters'],
                'tokenizer_sha256':original_sha,'saved_unicode_version':metadata.get('unicode_version'),
                'parent_sha256':evidence['parent_sha256'],'nonzero_backbone_gradients':True,'optimizer_steps':0,
                'source':source_hashes()}
        atomic_json(RUN_ROOT/('preflight_'+stage+'_'+str(int(time.time()))+'.json'),record)
        print('GPU PREFLIGHT PASSED:',stage,{k:record[k] for k in ('gpu','family','method','parameters','trainable_parameters')},flush=True)
        return record
    finally:
        del model;gc.collect();torch.cuda.empty_cache()

if not (RUN_ROOT/'complete.json').exists():
    main_check=_preflight(CONFIG,'student' if CONFIG['experiment']=='distillation' else 'model')
    if CONFIG['experiment']=='distillation':
        teacher_cfg=copy.deepcopy(CONFIG)
        teacher_cfg.update(family='d222',method='lorma',bundle=CONFIG['teacher_bundle'])
        teacher_check=_preflight(teacher_cfg,'teacher')
        assert teacher_check['parameters']>main_check['parameters'], 'Teacher must be larger than the A0 student'
        print('Independent teacher/student preflights passed. Teacher training runs first.',flush=True)
revision=Path(CONFIG['data_root'])/'download_revision.json'
if revision.exists():shutil.copy2(revision,RUN_ROOT/'official_download_revision.json')
print('LOCAL SQLITE CACHE:',Path(CONFIG['work_root'])/'cjpe_cache_local'/CONFIG['name'],flush=True)
print('PERSISTENT CHECKPOINTS:',RUN_ROOT,flush=True)
status(RUN_ROOT,'starting_worker',error=None,note='GPU checks passed; preparing local caches before optimizer training.')
try:
    RESULT=supervise(CONFIG)
except Exception:
    supervisor=RUN_ROOT/'supervisor.json'
    active=Path(json.loads(supervisor.read_text()).get('active_config',CONFIG)['run_root']) if supervisor.exists() else RUN_ROOT
    failure=active/'failure.json'
    if failure.exists():print('CURRENT FAILURE:\n'+json.loads(failure.read_text()).get('traceback',''))
    raise
official_ids=json.loads((Path(RESULT['run_root'])/'splits.json').read_text())['test']
checks=validate_zip(RESULT['submission'],official_ids)
assert checks['sha256']==RESULT['sha256']
READY=Path('/content/drive/MyDrive/CS781_Phase4/SUBMISSIONS') if Path('/content/drive/MyDrive').exists() else Path(CONFIG['run_root']).parents[1]/'SUBMISSIONS'
READY.mkdir(parents=True,exist_ok=True)
SUBMISSION=READY/Path(RESULT['submission']).name
if SUBMISSION.resolve()!=Path(RESULT['submission']).resolve():shutil.copy2(RESULT['submission'],SUBMISSION)
assert sha(SUBMISSION)==RESULT['sha256']
atomic_json(READY/(SUBMISSION.stem+'_RESULT.json'),RESULT)
print('SUBMISSION ZIP READY:',SUBMISSION,flush=True)
print('LOCAL DEV MACRO F1:',RESULT['dev_metrics']['macro_f1'],flush=True)
print('COMPLETED AFTER DEADLINE:',RESULT['completed_after_deadline'],flush=True)
display(FileLink(str(SUBMISSION)))
# The prediction ZIP is already safe before optional archive creation starts.
try:
    RECORDS=package_records(RESULT)
    print('MODEL/SOURCE RECORDS READY:',RECORDS,flush=True)
except Exception as error:
    print('Prediction ZIP remains safe. Optional records archive failed:',repr(error),flush=True)


In [ ]:
# Current status + exact result link. Run this after completion or interruption.
import json
from IPython.display import display,FileLink
supervisor_file=RUN_ROOT/'supervisor.json'
active=Path(json.loads(supervisor_file.read_text())['active_config']['run_root']) if supervisor_file.exists() else RUN_ROOT
for filename in ('status.json','failure.json','latest_export.json','complete.json'):
    path=active/filename
    if path.exists():print(filename+(' (historical failure record; see status/complete for current state)' if filename=='failure.json' else ''),json.loads(path.read_text()))
export=active/'latest_export.json'
if export.exists():
    result=json.loads(export.read_text());print('SUBMISSION ZIP:',result['submission']);display(FileLink(result['submission']))
    print('Colab: use the Files pane / mounted Drive to download this exact ZIP. Choose D:/CS781/Phase4_20261020/deliverables on Windows.')


In [ ]:
# Optional model-records archive AFTER the prediction ZIP is safely saved.
# This can take extra minutes and hundreds of MB; enable only when ready.
MAKE_MODEL_RECORDS=False
if MAKE_MODEL_RECORDS:
    from phase4_cjpe.export import package_records
    records=package_records(RESULT)
    print('Records only; NOT for Codabench:',records)


## Resume and failure handling

Rerun with the same source, data, input ZIP and CONFIG. The full latest.pt restores optimizer/scaler/RNG and the data cursor; best.pt is the selected task model. Historical Phase1 optimizer buffers are not reused for this new task.

Status files contain real optimizer step and last-update time. A still-open tab is not training evidence. The supervisor retries transient I/O failures up to twice and moves OOM recovery to a separate output folder with smaller chunks and explicitly recorded fresh-optimizer warm start. It does not repair unknown code/data errors or resurrect a revoked Colab runtime. Login/Drive/GPU access must be established while you are present.

Only upload UPLOAD_Phase4_*.zip containing root prediction.json. Local macro-F1 is not a hidden leaderboard score. Preserve source, checkpoints and authentic failure logs for the report. Do not use old dev or leaked test labels during fine-tuning.

Before downloading, enable the browser's Ask where to save downloads and select your D: folder. This notebook does not silently change your Windows download settings.

References: official CS781 capstone and IL-TUR; LoRMA (Bihany et al.2025), adapters (Houlsby et al.2019), BitFit (Ben Zaken et al.2022), knowledge distillation (Hinton et al.2015). Own original source is preserved in the embedded package. Record actual hardware and coding assistance honestly.


# B1 query/hierarchy improvement
Run the single cell with T4 selected. Original notebook stopped at Drive mount, before training. Complete Drive/dataset authentication using the account holding Phase3_B1_Input.zip.
Eight windows of up to 384 tokens; four learned decoder queries; encoder attention/mean and decoder feature fusion; small ordered document BiGRU; partial updates to the last two encoder/decoder layers. No score gain has been measured.
Four-hour training wall-time budget, additional preparation/export, automatic 75-minute deadline reserve. Only rolling latest.pt and selected best.pt are written in a new run. Other runs and inputs are untouched.


In [ ]:
# SINGLE B1 IMPROVEMENT CELL. Run this cell instead of the old training cells.
# Your own Phase1 B1 input is reused; new training goes to its own Drive folder.
import os,sys,time,json,hashlib,shutil,subprocess,importlib.util,base64,io,zipfile,gc
from pathlib import Path
for proc in Path('/proc').iterdir():
 if not proc.name.isdigit():continue
 try:args=proc.joinpath('cmdline').read_bytes().split(b'\0')
 except (FileNotFoundError,PermissionError,ProcessLookupError):continue
 if b'phase4_cjpe.runtime' in args and b'--worker' in args:raise RuntimeError('A Phase4 worker is already running here, PID '+proc.name+'. Do not start a second worker.')
for module,package in [('ijson','ijson>=3.2,<4'),('huggingface_hub','huggingface_hub>=0.26,<2'),('sklearn','scikit-learn>=1.4,<2'),('tokenizers','tokenizers')]:
 if importlib.util.find_spec(module) is None:subprocess.check_call([sys.executable,'-m','pip','install','-q',package])
import torch
assert torch.cuda.is_available(),'Select Runtime > Change runtime type > T4 GPU.'
print('ACTUAL GPU:',torch.cuda.get_device_name(0),flush=True)
from google.colab import drive
if not Path('/content/drive/MyDrive').is_dir():
 try:drive.mount('/content/drive',force_remount=True)
 except Exception as exc:raise RuntimeError('Drive mount failed. Complete the Google permission prompt with the account holding your B1 input. The model has not started training. Original error: '+str(exc)) from exc
assert Path('/content/drive/MyDrive').is_dir(),'Drive is not mounted; no training started.'
PERSIST=Path('/content/drive/MyDrive/CS781_Phase4')
WORK=Path('/content/phase4_work');WORK.mkdir(parents=True,exist_ok=True)
assert shutil.disk_usage('/content').free>12e9,'Need at least 12 GB free temporary Colab disk.'
RUN_ROOT=PERSIST/'runs/02_B1_Partial_QueryHier';RUN_ROOT.mkdir(parents=True,exist_ok=True)
INPUT_NAME='Phase3_B1_Input.zip'
INPUT=next((p for p in (PERSIST/'inputs'/INPUT_NAME,PERSIST/INPUT_NAME,Path('/content/drive/MyDrive')/INPUT_NAME,Path('/content/inputs')/INPUT_NAME,Path('/content')/INPUT_NAME) if p.is_file()),None)
assert INPUT is not None,'Upload the existing Phase3_B1_Input.zip to this account: MyDrive/CS781_Phase4/inputs/. Importing a notebook does not transfer its model weights.'
CODE_ROOT=Path('/content/phase4_b1_query_hierarchy_code')
source_bytes=base64.b64decode('...')
assert hashlib.sha256(source_bytes).hexdigest()=='360bfdaa23588b0893e597fe6f1056a078c8587af0cbe3d3361bcd59bf2c8897'
with zipfile.ZipFile(io.BytesIO(source_bytes)) as z:
 assert z.testzip() is None
 for name in z.namelist():
  p=Path(name);assert not p.is_absolute() and '..' not in p.parts
  dest=CODE_ROOT/p;dest.parent.mkdir(parents=True,exist_ok=True);dest.write_bytes(z.read(name))
sys.path[:]=[str(CODE_ROOT)]+[p for p in sys.path if p!=str(CODE_ROOT)]
for module in list(sys.modules):
 if module in ('phase4_cjpe','phase3_rr') or module.startswith(('phase4_cjpe.','phase3_rr.')):del sys.modules[module]
importlib.invalidate_caches()
from phase4_cjpe.common import sha,atomic_json,source_hashes,now
print('Checking the existing 1.49 GB B1 input SHA; this may take a few minutes on Drive.',flush=True)
assert INPUT.stat().st_size==1491952478 and sha(INPUT)=='22b49e8f370e8f9bfc1a3f3f88cab813c0e7443edb9091e30f8d4877508690c9','B1 input upload is incomplete or different.'
DATA_ROOT=WORK/'Phase_4_CJPE'
from phase4_cjpe.data import acquire
missing=any(not (DATA_ROOT/p).is_file() for p in ('Train/train.json','Train/dev.json','Test/test.json'))
data_zip=next((p for p in (PERSIST/'inputs/Phase_4_CJPE.zip',Path('/content/Phase_4_CJPE.zip')) if p.is_file()),None)
try:DATA_ROOT=acquire(DATA_ROOT,uploaded_zip=data_zip if missing else None,download=bool(missing and not data_zip))
except Exception as exc:
 if getattr(getattr(exc,'response',None),'status_code',None) not in (401,403):raise
 from getpass import getpass
 from huggingface_hub import login
 print('Complete official-dataset authentication before leaving this running.',flush=True)
 token=getpass('Approved Hugging Face READ token (hidden): ')
 try:login(token=token,add_to_git_credential=False)
 finally:del token
 DATA_ROOT=acquire(DATA_ROOT,download=True)
CONFIG={'name':'02_B1_Partial_QueryHier','family':'b1','method':'partial','experiment':'adaptation',
 'rank':8,'alpha':16,'seed':781,'max_tokens':384,'chunks_per_case':8,'region_chars':8192,
 'max_train_cases':12000,'holdout_per_class':256,'max_epochs':4,'accumulation':8,
 'backbone_lr':2e-5,'head_lr':3e-4,'training_hours':4.,'eval_every':100,'save_every':25,'patience':6,
 'teacher_hours':1.5,'kd_temperature':2.,'kd_weight':.5,'stall_minutes':45,
 'export_reserve_minutes':75,'deadline_utc':'2026-10-21T18:29:00Z','allow_after_deadline':False,
 'run_root':str(RUN_ROOT),'work_root':str(WORK),'data_root':str(DATA_ROOT),'bundle':str(INPUT),
 'chunk_batch_size':2,'document_architecture':'b1_query_hierarchy_v1','label_smoothing':.02}
# On an exact resume keep the previously selected memory setting unchanged.
saved=RUN_ROOT/'worker_config.json'
if saved.exists():
 prior=json.loads(saved.read_text())
 assert {k:v for k,v in prior.items() if k!='chunk_batch_size'}=={k:v for k,v in CONFIG.items() if k!='chunk_batch_size'},'Run configuration changed. Preserve this run and choose a new run name.'
 CONFIG['chunk_batch_size']=prior['chunk_batch_size']
from phase4_cjpe.train import deadline_timestamp
if not (RUN_ROOT/'complete.json').exists():
 assert time.time()<deadline_timestamp(CONFIG)-CONFIG['export_reserve_minutes']*60,'Official training cutoff passed. No on-time submission will be fabricated.'
revision=DATA_ROOT/'download_revision.json'
if revision.exists():shutil.copy2(revision,RUN_ROOT/'official_download_revision.json')
atomic_json(RUN_ROOT/'design_and_retention.json',{'architecture':CONFIG['document_architecture'],
 'query_count':4,'encoder_layers_updated':2,'decoder_layers_updated':2,'windows':8,'tokens_per_window':384,
 'retention':'Only rolling latest.pt full resume and selected best.pt; atomic replacements, no numbered checkpoint history.',
 'original_input_and_other_runs_preserved':True,'source':source_hashes(),'recorded_utc':now()})

from phase3_rr.archive_inputs import resolve_bundle
from phase3_rr.data import Tokenizer
from phase4_cjpe.models import make_model
bundle,manifest=resolve_bundle(INPUT,WORK/'weights');assert manifest['family']=='b1'
tok=Tokenizer(bundle/'tokenizer.json')
sample='The Supreme Court considers the appeal under section १२३.'
ids=tok.encode(sample)
assert ids and tok.impl.decode(ids)==sample,'Original Unicode13 tokenizer round-trip failed.'

def gpu_preflight(batch_size):
 model=logits=loss=scaler=None
 try:
  model,evidence=make_model(bundle,'b1','partial',8,16);model.chunk_batch_size=batch_size
  raw=(ids*((382+len(ids)-1)//len(ids)))[:382]
  chunks=[[1]+raw+[2] for _ in range(8)];chunks[-1]=chunks[-1][:-64]+[2]
  tensor_bytes=sum(t.numel()*t.element_size() for t in model.state_dict().values())
  trainable_bytes=sum(p.numel()*p.element_size() for p in model.parameters() if p.requires_grad)
  peak=int(3*tensor_bytes+4*trainable_bytes+250e6)
  print('Estimated additional peak Drive space for atomic saves:',round(peak/1e9,2),'GB. Account quota is not independently verified.',flush=True)
  model=model.to('cuda').train()
  with torch.autocast(device_type='cuda',dtype=torch.float16):logits=model(chunks,[i/7 for i in range(8)],torch.device('cuda')).float()
  loss=torch.nn.functional.cross_entropy(logits,torch.tensor([0],device='cuda'))
  assert bool(torch.isfinite(loss)),'Nonfinite GPU preflight loss.'
  scaler=torch.amp.GradScaler('cuda',init_scale=128);scaler.scale(loss).backward()
  for group,prefixes in [('encoder',('backbone.encoder.',)),('decoder',('backbone.decoder.',)),('new head',('queries','token_pool.','chunk_projection.','document_encoder.','document_pool.','head.'))]:
   found=False
   for name,param in model.named_parameters():
    if name.startswith(prefixes) and param.requires_grad and param.grad is not None:
     assert bool(torch.isfinite(param.grad).all()),'Nonfinite '+group+' gradient: '+name
     if bool(param.grad.abs().max()>0):found=True
   assert found,'No nonzero '+group+' gradients.'
  result={'actual_gpu':torch.cuda.get_device_name(0),'finite_loss':float(loss.detach()),'parameters':evidence['parameters'],
   'trainable_parameters':evidence['trainable_parameters'],'chunk_batch_size':batch_size,'windows':8,'tokens_per_window':384,
   'encoder_decoder_and_head_gradients_verified':True,'estimated_additional_drive_peak_bytes':peak,
   'drive_quota_verified':False,'optimizer_steps':0,'checked_utc':now()}
  return result
 finally:
  del model,logits,loss,scaler;gc.collect();torch.cuda.empty_cache()

if not (RUN_ROOT/'complete.json').exists():
 print('Checking all eight 384-token windows on the real GPU before training.',flush=True)
 memory_fallback=False
 try:preflight=gpu_preflight(CONFIG['chunk_batch_size'])
 except torch.OutOfMemoryError:
  if CONFIG['chunk_batch_size']==1 or saved.exists():raise
  memory_fallback=True
 if memory_fallback:
  gc.collect();torch.cuda.empty_cache()
  CONFIG['chunk_batch_size']=1
  print('Preflight memory fallback: one chunk at a time, preserving all eight windows and context length.',flush=True)
  preflight=gpu_preflight(1)
 atomic_json(RUN_ROOT/'preflight.json',preflight)
 print('TOKENIZER + FULL-CONTEXT GPU PREFLIGHT PASSED:',preflight,flush=True)
print('Checkpoint policy: replace latest.pt and best.pt atomically; no epoch/step checkpoint history.',flush=True)
print('Preparing local SQLite caches, then training. Confirm increasing optimizer steps before leaving.',flush=True)
from phase4_cjpe.runtime import supervise
from phase4_cjpe.export import validate_zip
try:RESULT=supervise(CONFIG)
except Exception:
 failure=RUN_ROOT/'failure.json'
 if failure.exists():print('CURRENT FAILURE:\n'+json.loads(failure.read_text()).get('traceback',''))
 raise
expected_ids=json.loads((RUN_ROOT/'splits.json').read_text())['test']
checks=validate_zip(RESULT['submission'],expected_ids);assert checks['sha256']==RESULT['sha256']
READY=PERSIST/'SUBMISSIONS';READY.mkdir(parents=True,exist_ok=True)
SUBMISSION=READY/Path(RESULT['submission']).name
shutil.copy2(RESULT['submission'],SUBMISSION);assert sha(SUBMISSION)==RESULT['sha256']
atomic_json(READY/(SUBMISSION.stem+'_RESULT.json'),RESULT)
print('VERIFIED SUBMISSION ZIP:',SUBMISSION,flush=True)
print('LOCAL DEV MACRO F1:',RESULT['dev_metrics']['macro_f1'],flush=True)
print('NEW OPTIMIZER UPDATES:',RESULT['optimizer_updates'],flush=True)
print('COMPLETED AFTER DEADLINE:',RESULT['completed_after_deadline'],flush=True)
from IPython.display import FileLink,display
display(FileLink(str(SUBMISSION)))
# No duplicate model-weights archive: selected weights remain in best.pt.
records=RUN_ROOT/'EXACT_SOURCE_B1_QUERY_HIERARCHY.zip';records.write_bytes(source_bytes)
assert sha(records)==hashlib.sha256(source_bytes).hexdigest()
print('BEST MODEL:',RUN_ROOT/'best.pt',flush=True)
print('FULL OPTIMIZER RESUME:',RUN_ROOT/'latest.pt',flush=True)
print('Source, tokenizer, data audit, exact splits, selection, logs and probabilities are preserved in:',RUN_ROOT,flush=True)
